# Byte I/O - JavaScript

All 26 JavaScript examples from [docs/io.md](https://platob.github.io/yggdryl/io/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and call `require`, so they need a CommonJS
JavaScript kernel such as
[IJavascript](https://github.com/n-riesco/ijavascript), with the package
installed beside the notebook:

```console
npm install yggdryl
```

In [ ]:
const assert = require('node:assert/strict')
const { IOBase } = require('yggdryl')

const handle = IOBase.fromBytes()
handle.pwrite(0, Buffer.from('symbol,price\n'))
handle.pwrite(13, Buffer.from('AAPL,1\n'))
assert.equal(handle.size, 20)

// Two reads at different offsets, in any order: there is no shared cursor.
assert.equal(handle.pread(13, 4).toString(), 'AAPL')
assert.equal(handle.pread(0, 6).toString(), 'symbol')

## Laziness

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))

// Constructing touches nothing: no file is created, opened, or mapped.
const handle = new IOBase(path.join(root, 'nested', 'lazy.csv'))
assert.ok(!handle.exists())

// Reading something absent yields nothing rather than throwing.
assert.equal(handle.size, 0)
assert.equal(handle.readBytes().length, 0)

// Writing creates the resource, and any parent it needs.
handle.writeText('symbol,price\n')
assert.ok(handle.isFile())
assert.equal(handle.readText(), 'symbol,price\n')

fs.rmSync(root, { recursive: true, force: true })

## Kinds

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const folder = new IOBase(fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-')))
assert.ok(folder.isDir())
assert.ok(!folder.isFile())

// Nothing is there, so nothing has decided; a write settles it.
const leaf = folder.joinpath('ticks.csv')
assert.ok(!leaf.exists())
leaf.writeText('symbol\n')
assert.ok(leaf.isFile())
assert.ok(!leaf.isDir())

fs.rmSync(folder.toPath(), { recursive: true, force: true })

## Bytes or rows

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const root = new IOBase(fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-')))

const notes = root.joinpath('notes.txt')
assert.ok(notes.isAtomic())
assert.ok(!notes.isTabular())

// The name is enough: nothing has been written to this location yet.
const trades = root.joinpath('trades.parquet')
assert.ok(trades.isTabular())
assert.ok(!trades.isAtomic())

assert.ok(!root.isAtomic())

fs.rmSync(root.toPath(), { recursive: true, force: true })

## Whole values

In [ ]:
const assert = require('node:assert/strict')
const { IOBase } = require('yggdryl')

const handle = IOBase.fromBytes()
handle.writeBytes(Buffer.from('symbol,price\n'))

// `append` reports the offset the bytes landed at.
assert.equal(handle.append(Buffer.from('AAPL,1\n')), 13)
assert.equal(handle.pread(0, 6).toString(), 'symbol')
// A range past the end yields what exists rather than throwing.
assert.equal(handle.pread(100, 4).length, 0)
assert.equal(handle.readBytes().length, 20)

## Cursors

In [ ]:
const assert = require('node:assert/strict')
const { IOBase } = require('yggdryl')

const handle = IOBase.fromBytes()
const cursor = handle.cursor()
cursor.write(Buffer.from('symbol,price\n'))

assert.equal(handle.readBytes().toString(), 'symbol,price\n')
cursor.seek(7)
assert.equal(cursor.read(5).toString(), 'price')

## Text records

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const zlib = require('node:zlib')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const target = path.join(root, 'trades.jsonl.gz')
fs.writeFileSync(target, zlib.gzipSync('{"id":1}\r\n{"id":2}\n'))

assert.deepEqual([...new IOBase(target).readLines()], ['{"id":1}', '{"id":2}'])

// Pinned, a lone `\n` is content rather than a break.
const mixed = path.join(root, 'mixed.txt')
fs.writeFileSync(mixed, 'lf\ncrlf\r\nlast')
assert.deepEqual([...new IOBase(mixed).readLines({ linesep: '\\r\\n' })], [
  'lf\ncrlf',
  'last',
])

fs.rmSync(root, { recursive: true, force: true })

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const target = path.join(root, 'app.log')
fs.writeFileSync(
  target,
  '2024-02-01 10:00:00.000_000 [ee] [alpha] boom\n' +
    '  at frame one\n' +
    '2024-02-01 10:00:01.000_000 [ii] [beta] fine\n',
)

const entries = [...new IOBase(target).readLines('^\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2}')]
assert.deepEqual(entries, [...new IOBase(target).readLines({ logs: true })])
assert.equal(entries.length, 2)
assert.equal(entries[0], '2024-02-01 10:00:00.000_000 [ee] [alpha] boom\n  at frame one')

fs.rmSync(root, { recursive: true, force: true })

### Writing records

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const handle = new IOBase(path.join(root, 'out.log'))
// A generator, never an array the binding materializes first.
handle.writeLines(
  (function* rows() {
    for (let index = 0; index < 1_000; index += 1) {
      yield `row-${index}`
    }
  })(),
)
handle.appendLines(['tail'])
assert.ok(handle.readBytes().toString().endsWith('row-999\ntail\n'))
assert.equal([...handle.readLines()].length, 1_001)

const pinned = new IOBase(path.join(root, 'crlf.log'))
pinned.writeLines(['one', 'two'], { linesep: '\\r\\n' })
assert.equal(pinned.readBytes().toString(), 'one\r\ntwo\r\n')

fs.rmSync(root, { recursive: true, force: true })

### Records as Arrow batches

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const zlib = require('node:zlib')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const target = path.join(root, 'app.log.gz')
fs.writeFileSync(target, zlib.gzipSync(
  '2024-02-01 10:00:00.000_000 [ee] [alpha] boom\n    at frame one\n' +
    '2024-02-01 10:00:01.500 [ii] [beta] fill 100 @ 187.23\n',
))

const pattern =
  '^\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2}\\S* \\[(?<level>[^\\]]+)\\] \\[(?<logger>[^\\]]+)\\]'
const table = new IOBase(target)
  .readArrowLines(pattern, { customFields: { venue: 'XNAS' } })
  .toTable()
assert.equal(table.numRows, 2)
assert.deepEqual([...table.getChild('level')], ['ee', 'ii'])
assert.equal([...table.getChild('unix')][0], 1_706_781_600_000_000_000n)
assert.deepEqual([...table.getChild('venue')], ['XNAS', 'XNAS'])

fs.rmSync(root, { recursive: true, force: true })

### A reader is a configuration document

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase, schemaFromPattern, yaml } = require('yggdryl')

const options = yaml.loads(
  [
    "pattern: '^(?<stamp>\\S+) \\[(?<level>[A-Z]+)\\]'",
    'byte_size: 1048576',
    'batch_size: 4096',
    'timestamp_capture: stamp',
    'custom_fields:',
    '  source: gateway',
  ].join('\n'),
)

// The schema answers from the document alone, with no resource in sight.
const schema = schemaFromPattern(options)
assert.equal(String(schema.dataType.get('source').dataType), 'utf8')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const target = path.join(root, 'app.log')
fs.writeFileSync(target, '2024-02-01T10:00:00 [ERROR] boom\n')
const table = new IOBase(target).readArrowLines(options).toTable()
assert.deepEqual([...table.getChild('source')], ['gateway'])

fs.rmSync(root, { recursive: true, force: true })

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase, schemaFromPattern } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const target = path.join(root, 'app.log')
fs.writeFileSync(target, '2024-02-01 10:00:00 [42] (info) qty=1.50 fill\n')

const pattern =
  '^\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2} \\[(?<thread_id>\\d+)\\] \\((?<log_level>\\w+)\\) qty=(?<qty>[0-9.]+)'

// The standalone builder answers the emitted root without a reader.
const schema = schemaFromPattern(pattern, { captureTypes: { qty: 'decimal(9, 2)' } })
assert.equal(String(schema.dataType.get('thread_id').dataType), 'int64')

const table = new IOBase(target)
  .readArrowLines(pattern, { captureTypes: { qty: 'decimal(9, 2)' } })
  .toTable()
assert.deepEqual([...table.getChild('thread_id')], [42n])
assert.deepEqual([...table.getChild('log_level')], ['info'])

fs.rmSync(root, { recursive: true, force: true })

### Trimming, and a zone that makes `unix` an instant

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const target = path.join(root, 'app.log')
fs.writeFileSync(target, '2024-02-01T00:00:00 [INFO] x\n')
const pattern = '^(?<stamp>\\S+) \\[(?<level>[A-Z]+)\\]'

const source = new IOBase(target)
const naive = [...source.readArrowLines(pattern).toTable().getChild('unix')]
const zoned = [
  ...source
    .readArrowLines(pattern, { timezone: '+02:00', rstrip: 'ascii' })
    .toTable()
    .getChild('unix'),
]
assert.equal(naive[0], 1_706_745_600_000_000_000n)
assert.equal(zoned[0], naive[0] - 2n * 3_600n * 1_000_000_000n)

fs.rmSync(root, { recursive: true, force: true })

### Streaming a trading log into a partitioned Iceberg table

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const zlib = require('node:zlib')
const { IOBase, iceberg } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-doc-'))
const logs = path.join(root, 'incoming')
fs.mkdirSync(logs)
fs.writeFileSync(
  path.join(logs, 'a.log'),
  '2024-02-01 10:00:00.000_000 [ee] [alpha] boom\n    at frame one\n' +
    '2024-02-01 10:00:01.000_000 [ii] [beta] fill 100 @ 187.23\n',
)
fs.writeFileSync(
  path.join(logs, 'b.log.gz'),
  zlib.gzipSync('2024-02-01 11:00:00.000_000 [ii] [gamma] fill 200 @ 188.01\n'),
)

const pattern =
  '^\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2}\\S* \\[(?<level>[^\\]]+)\\] \\[(?<logger>[^\\]]+)\\]'
const reader = new IOBase(logs).readArrowLines(pattern, { customFields: { venue: 'XNAS' } })

// The reader's root field is the table's schema; marking `level` partitions it.
const catalog = new iceberg.Catalog(path.join(root, 'warehouse'))
catalog.createTable('logs.app', reader.field.withPartitionFields(['level']))

// The append consumes the parse itself - lazy, one batch at a time.
const table = catalog.append('logs.app', reader)
const snapshot = table.currentSnapshot
assert.equal(snapshot.operation, 'append')
assert.equal(snapshot.summary['added-records'], '3')
assert.equal(snapshot.summary['added-data-files'], '2') // one file per level

const again = catalog.append(
  'logs.app',
  new IOBase(path.join(logs, 'a.log')).readArrowLines(pattern, {
    customFields: { venue: 'XNAS' },
  }),
)
assert.equal(again.snapshots.length, 2)
// a.log holds two records, so the totals accumulate 3 + 2.
assert.equal(again.currentSnapshot.summary['total-records'], '5')
assert.equal(again.scan().toTable().numRows, 5)

fs.rmSync(root, { recursive: true, force: true })

## What the bytes are

In [ ]:
const assert = require('node:assert/strict')
const { IOBase, MimeType } = require('yggdryl')

// Nothing names an in-memory buffer, so its type comes from its bytes.
const handle = IOBase.fromBytes(Buffer.from('{"symbol":"AAPL"}'))
assert.ok(handle.mediaType.base.equals(MimeType.JSON))

// It is re-derived after the content changes.
handle.writeBytes(Buffer.from('PAR1payload'))
assert.ok(handle.mediaType.base.equals(MimeType.PARQUET))

## Adding and removing a coding

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const plain = new IOBase(path.join(root, 'rows.json'))
plain.writeBytes(Buffer.from('{"symbol":"AAPL"}'))

// Nothing wraps these bytes, so there is nothing to undo.
assert.equal(plain.codec, null)

const encoded = new IOBase(path.join(root, 'rows.json.gz'))
assert.equal(encoded.codec, 'gzip')

// The target's name already said gzip, so nothing here repeats it.
assert.equal(plain.compressInto(encoded), encoded.size)
assert.deepEqual([...encoded.readBytes().subarray(0, 2)], [0x1f, 0x8b])

const decoded = new IOBase(path.join(root, 'roundtrip.json'))
assert.equal(encoded.decompressInto(decoded), 17)
assert.equal(decoded.readText(), '{"symbol":"AAPL"}')
assert.equal(decoded.codec, null)

// An in-memory target has no name to declare a coding, so this one is named.
const memory = IOBase.fromBytes()
assert.ok(plain.compressInto(memory, 'zstd') > 0)
assert.equal(memory.codec, 'zstd')

// A target declaring no coding is refused rather than copied unchanged.
assert.throws(
  () => plain.compressInto(new IOBase(path.join(root, 'copy.json'))),
  /expected a target declaring a content coding/,
)
assert.equal(fs.existsSync(path.join(root, 'copy.json')), false)

fs.rmSync(root, { recursive: true, force: true })

## Clearing and removing

In [ ]:
const assert = require('node:assert')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const root = path.join(os.tmpdir(), `yggdryl-docs-lifecycle-${process.pid}`)
const handle = new IOBase(path.join(root, 'logs'))
handle.mkdir()
handle.joinpath(['a.log']).writeText('line\n')

// Clearing empties the container and keeps it.
handle.clear()
assert.equal([...handle.ls(true, false)].length, 0)

// Removing deletes it; a second call succeeds, having done nothing.
handle.remove()
handle.remove()
assert.equal([...new IOBase(root).ls(false, false)].length, 0)

// The handle stays usable and lazy - a write recreates the resource.
const leaf = new IOBase(path.join(root, 'trades.csv'))
leaf.writeText('symbol,price\n')
leaf.remove()
assert.equal(leaf.exists(), false)
leaf.writeText('symbol,price\n')
assert.equal(leaf.readText(), 'symbol,price\n')
new IOBase(root).remove(true)

## Arrow batches

In [ ]:
const assert = require('node:assert/strict')
const arrow = require('apache-arrow')
const { BatchReader, Field, IOBase, MimeType, fields } = require('yggdryl')

// A non-null struct Field is the schema.
const schema = fields.struct(
  'row',
  [Field.from('id: int64'), Field.from('symbol: utf8')],
  { nullable: false },
)

const table = new arrow.Table({
  id: arrow.vectorFromArray([1n, 2n], new arrow.Int64()),
  symbol: arrow.vectorFromArray(['AAPL', null], new arrow.Utf8()),
})

// The handle's own media type picks the encoding; no format argument is passed.
const handle = IOBase.fromBytes()
handle.mediaType = MimeType.ARROW_STREAM
const options = handle.recordOptions()

// The write path takes a batch reader and nothing else.
handle.writeArrowBatchReader(BatchReader.from(table), options)
assert.ok(handle.readArrowField(options).equals(schema))

// The read path returns one. Batches arrive one at a time, never as a vector.
let rows = 0
for (const batch of handle.readArrowBatchReader(options)) {
  rows += batch.numRows
}
assert.equal(rows, 2)

In [ ]:
const assert = require('node:assert/strict')
const { IOBase, MimeType } = require('yggdryl')

// An absent resource holds no batches rather than failing to parse.
const empty = IOBase.fromBytes()
empty.mediaType = MimeType.ARROW_STREAM
assert.equal([...empty.readArrowBatchReader()].length, 0)

// An encoding this build does not implement is named rather than guessed.
const csv = IOBase.fromBytes()
csv.mediaType = MimeType.CSV
assert.throws(() => csv.recordOptions(), /text\/csv/)

## Rows as record instances

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const handle = new IOBase(path.join(root, 'trades.arrows'))
// Plain objects are rows; `writeRecords` is the generic write under
// the record name.
handle.writeRecords([
  { id: 1n, venue: 'XNAS' },
  { id: 2n, venue: null },
])

// Plain objects out, streamed batch by batch ...
assert.deepEqual([...handle.readRecords()].map((row) => row.id), [1n, 2n])

// ... or instances of any class whose constructor takes the plain row.
class Trade {
  constructor(row) {
    Object.assign(this, row)
  }
}
const trades = [...handle.readRecords(Trade)]
assert.ok(trades.every((t) => t instanceof Trade))

// An absent resource yields no records rather than raising.
assert.deepEqual([...new IOBase(path.join(root, 'absent.arrows')).readRecords()], [])

fs.rmSync(root, { recursive: true, force: true })

## Column pushdown

In [ ]:
const assert = require('node:assert/strict')
const arrow = require('apache-arrow')
const { BatchReader, Field, IOBase, MimeType, fields } = require('yggdryl')

const table = new arrow.Table({
  id: arrow.vectorFromArray([1n, 2n], new arrow.Int64()),
  symbol: arrow.vectorFromArray(['AAPL', 'MSFT'], new arrow.Utf8()),
  venue: arrow.vectorFromArray(['XNAS', 'XNAS'], new arrow.Utf8()),
})

const handle = IOBase.fromBytes()
handle.mediaType = MimeType.ARROW_STREAM
handle.writeArrowBatchReader(BatchReader.from(table))

// One of the three columns, declared as this read's schema.
const wanted = fields.struct('row', [Field.from('id: int64')], { nullable: false })
const options = handle.recordOptions()

const projected = handle.readArrowBatchReader(options.withSchema(wanted))
assert.equal(projected.field.dataType.length, 1)
assert.equal(projected.toTable().numCols, 1)

// The resource is unchanged: it still holds all three.
assert.equal(handle.readArrowField().dataType.length, 3)

// A column it does not hold cannot be projected out of it, so the encoding
// reads everything and the cast supplies that column as nulls.
const invented = fields.struct(
  'row',
  [Field.from('id: int64'), Field.from('nowhere: utf8?')],
  { nullable: false },
)
const widened = handle.readArrowBatchReader(options.withSchema(invented))
assert.equal(widened.field.dataType.length, 2)

## Appending and merging

In [ ]:
const assert = require('node:assert/strict')
const arrow = require('apache-arrow')
const { BatchReader, Field, IOBase, MimeType, fields } = require('yggdryl')

const schema = fields.struct(
  'row',
  [Field.from('id: int64'), Field.from('symbol: utf8?')],
  { nullable: false },
)
const rows = (ids, symbols) =>
  BatchReader.from(
    new arrow.Table({
      id: arrow.vectorFromArray(ids, new arrow.Int64()),
      symbol: arrow.vectorFromArray(symbols, new arrow.Utf8()),
    }),
  )

const handle = IOBase.fromBytes()
handle.mediaType = MimeType.ARROW_STREAM
const options = handle.recordOptions().withSchema(schema)

// No match key: the resource is replaced.
handle.writeArrowBatchReader(rows([1n, 2n], ['AAPL', 'MSFT']), options)

// Appending reads what is there, chains the new batches after it, and rewrites.
handle.appendArrowBatchReader(rows([3n], ['NVDA']), options)
assert.equal(handle.readArrowBatchReader(options).toTable().numRows, 3)

// A match key merges: `2` is already stored and updates, `9` is new and appends.
const merging = options.withMergeByNames(['id'])
handle.writeArrowBatchReader(rows([2n, 9n], ['MSFT.O', 'AMD']), merging)
assert.equal(handle.readArrowBatchReader(options).toTable().numRows, 4)

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const handle = new IOBase(path.join(root, 'orders.arrows'))
handle.writeArrowBatchReader(
  new arrow.Table({
    id: arrow.vectorFromArray([1n, 2n], new arrow.Int64()),
    symbol: arrow.vectorFromArray(['AAPL', 'MSFT'], new arrow.Utf8()),
  }),
)

const narrowed = handle.recordOptions().withSelectByNames(['symbol'])
const table = handle.readArrowBatchReader(narrowed).toTable()
assert.deepEqual(table.schema.fields.map((field) => field.name), ['symbol'])

fs.rmSync(root, { recursive: true, force: true })

## Globbing and Hive partitions

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const root = path.join(fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-')), 'lake')
for (const year of ['2024', '2025']) {
  const leaf = path.join(root, `year=${year}`, 'month=01')
  fs.mkdirSync(leaf, { recursive: true })
  fs.writeFileSync(path.join(leaf, 'part-0.parquet'), 'parquet')
}

const lake = new IOBase(root)

// A fixed prefix is descended, not listed and filtered.
assert.equal([...lake.glob('year=2024/**/*.parquet')].length, 1)
assert.equal([...lake.rglob('*.parquet')].length, 2)

// Partition filters select the leaves to overwrite or upsert.
const selected = [...lake.childrenWhere({ year: '2024' })]
assert.equal(selected.length, 1)
assert.deepEqual(selected[0].partitions, [
  { column: 'year', value: '2024' },
  { column: 'month', value: '01' },
])

fs.rmSync(root, { recursive: true, force: true })

## Partition pruning and filtering

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const lake = path.join(root, 'lake')
new IOBase(path.join(lake, 'year=2024', 'month=01', 'trades.arrows'))
  .writeRecords([{ id: 1n }, { id: 2n }])
new IOBase(path.join(lake, 'year=2024', 'month=02', 'trades.arrows'))
  .writeRecords([{ id: 3n }])

const handle = new IOBase(lake)
const options = handle.recordOptions().withFilterPartitions([
  ['year', '2024'],
  ['month', '01'],
])
assert.equal(handle.readArrowBatchReader(options).toTable().numRows, 2)

fs.rmSync(root, { recursive: true, force: true })

## Partition columns in the data

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { BatchReader, Field, IOBase, MimeType, RecordOptions, fields } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
fs.mkdirSync(path.join(root, 'year=2024', 'month=01'), { recursive: true })

const schema = fields.struct(
  'row',
  [Field.from('price: int64'), Field.from('year: int32'), Field.from('month: utf8')],
  { nullable: false },
)
const table = new arrow.Table({
  price: arrow.vectorFromArray([10n, 20n], new arrow.Int64()),
  year: arrow.vectorFromArray([2024, 2024], new arrow.Int32()),
  month: arrow.vectorFromArray(['01', '01'], new arrow.Utf8()),
})

// The rows carry every column; the write drops the two the path spells out.
const lake = new IOBase(root)
const options = RecordOptions.forMimeType(MimeType.ARROW_STREAM).withSchema(schema)
lake.writeArrowBatchReader(BatchReader.from(table), options)

// Only `price` reached the leaf; the other two are the directory names.
const leaf = lake.joinpath('year=2024').joinpath('month=01').joinpath('part-0.arrows')
assert.equal(leaf.readArrowField().dataType.length, 1)

// Reading the folder restores them with their declared types.
const restored = lake.readArrowBatchReader(options).toTable()
assert.equal(restored.numCols, 3)
assert.equal(restored.schema.fields[1].type.toString(), 'Int32')
assert.deepEqual(restored.getChild('month').toArray(), ['01', '01'])

fs.rmSync(root, { recursive: true, force: true })